In [68]:
!pip install -q ipdb
!pip install -1 tqdm
!pip install -q sentencepiece
!pip install -1 wandb


Usage:   
  pip3 install [options] <requirement specifier> [package-index-options] ...
  pip3 install [options] -r <requirements file> [package-index-options] ...
  pip3 install [options] [-e] <vcs project url> ...
  pip3 install [options] [-e] <local project path> ...
  pip3 install [options] <archive url/path> ...

no such option: -1

Usage:   
  pip3 install [options] <requirement specifier> [package-index-options] ...
  pip3 install [options] -r <requirements file> [package-index-options] ...
  pip3 install [options] [-e] <vcs project url> ...
  pip3 install [options] [-e] <local project path> ...
  pip3 install [options] <archive url/path> ...

no such option: -1


In [69]:
import ipdb
def calculate(v1,v2):
  # ipdb.set_trace()
  calc1=v1*4
  calc2=v2*3
  return calc1+calc2

print(calculate(2,3))

17


# **BUILDING LLMS FROM SCRATCH : BY UDEMY HAVIER IDEAMI**

In [ ]:

#coding start for the LLM :

1) IMPORT LIBRARIES

In [70]:
import os,sys
import ipdb # for debugging , variation of pdb
from tqdm import tqdm # check the progress of the training
from datetime import datetime # when naming the checkpoint files , like stop training and continue then so save the checkpoints= intermediate points of the training and dsave them witht the data
import platform, shutil
import requests,zipfile, io
#pytorch
import torch
import torch.nn as nn
from torch.nn import functional as F

#tokenizer : when train LLM use a dataset which have normal human text but AI architeccture work with number so translate the text to numbers , tokenizer split the text to tokens and assign the numbers to the token
import sentencepiece as spm

#this imooroves the performance as some kinds of gpu benefit with

torch.backends.cuda.matmul.allow_tf32=True
torch.backends.cudnn.allow_tf32=True

torch.cuda.empty_cache()

# ***DOWNLOAD FILES FOR THE PROJECT***




In [71]:
files_url="https://ideami.com/llm_train"
print("downloading files s=using the python")
response=requests.get(files_url)
zipfile.ZipFile(io.BytesIO(response.content)).extractall(".")

downloading files s=using the python


# **SETTING PARAMETERS OF THE ARCHITECTURE :**

In [72]:
#ARCHITECTURE_PARAMETERS
#we will be training in batches
# in ipad there is explanation
# take data and divide in samples as our objective is that predict the next word from

batch_size =   8
context = 512 # attention mechanism ke kitne context se relation or attention
embed_size=384
n_layers=7
n_heads=7
BIAS=True


#-------------------------
#hyperparameters:  impact our training process


lr=3e-4
dropout=0.05 # regularization 5% explained in ipad
weight_decay=0.01 #prevent the scales of the parameters to be smalla and prevent network adapting too tightly to network data
grad_clip=1.0

#--------------------------
# training iterations:

train_iters= 100000
eval_interval=50            # when training dataset so dont use all data for training, take % of data for training and some % for evaluation , evaluation data is unseen to model until final testing phase
eval_iters=10
compile=False #pytorch do processes with us and accelarate the training process

load_pretrained=False
checkpoint_dir="models/"
checkpoint_fn="latest.pt"       #fn=filename
checkpoint_load_fn="latest.pt"
dtype=torch.bfloat16
#mode
inference=False
device="cuda" if torch.cuda.is_available() else "cpu"
print("Device:",device)




Device: cuda


LOGGING

In [73]:
wandb_log=True
wandb_project="llm9" #
wandb_run_name="llm1-"+datetime.now().strftime("%Y_%m_%d_%H_%M_%S")

if wandb_log:
  import wandb
  wandb.init(project=wandb_project,name=wandb_run_name)


# NOW LOADING DATASET AND TRAINING LLM


In [74]:
with open("wiki.txt", "r" ,encoding="utf-8") as f:
  text=f.read()
print(text[10000:10301])

 that was used to represent a team in an old TV show, The A-Team. A capital a is written "A". Use a capital A at the start of a sentence if writing.

A is also a musical note, sometimes referred to as "La".

The letter 'A' was in the Phoenician alphabet's aleph. This symbol came from a simple picture


In [75]:
#tokenizer

sp=spm.SentencePieceProcessor(model_file="wiki_tokenizer.model") #activating the tokenizer

vocab_size=sp.get_piece_size()
print(f"Tokenizer vocal size:{vocab_size}")


Tokenizer vocal size:4096


In [76]:
encode =  lambda s:sp.Encode(s)        #to convert our text to tokens and onto numbers

decode =   lambda l:sp.Decode(l)       #to go from numbers to the text

print(encode("once upon a time"))

[2686, 698, 265, 261, 684]


In [77]:
print(decode(encode("once upon a time")))

once upon a time


In [78]:
if os.path.exists(f"encoded_data.pt"):
  data=torch.load("encoded_data.pt")
else:
  data=torch.tensor(encode(text),dtype=torch.long)
  torch.save(data,"encoded_data.pt")

In [79]:
data_size=len(data)
#split data in train and validation

spl=int(0.9*data_size)
train_data=data[:spl]
val_Data=data[spl:]

print(f"Total data: {data_size/1e6:.2f}Million | training: {len(train_data)/1e6:.2f} Million | Validation:{len(val_Data)/1e6:.2f}Million")
#divide by miilion for formatting and 2 decimal places

Total data: 59.21Million | training: 53.29 Million | Validation:5.92Million


In [80]:
#we divide the data into the batches and batch is the group of the training examples that is processed in one forward or back pass

def get_batch(split):
  data=train_data if split=="train" else val_Data
  inds=torch.randint(len(data)-context,(batch_size,))
  x=torch.stack([data[i:i+context] for i in inds])  #(bs,sl) -> (8,512)  8 sequences and each having 512 tokens -> so its input to our llm
  y=torch.stack([data[i+1:i+context+1] for i in inds])  # +1 -> because objective of llm to teach to predict next token so we pair each of token of each sequence with the token it should predict aka the next after it so true label is the next token

  x,y=x.to(device),y.to(device)
  return x,y

x,y=get_batch("train")
print(x.shape,y.shape)
print(x[0][:10])
print(y[0][0:10])




torch.Size([8, 512]) torch.Size([8, 512])
tensor([1912, 2625,  369,  860, 4051,  379,  307, 2939, 1998,  376],
       device='cuda:0')
tensor([2625,  369,  860, 4051,  379,  307, 2939, 1998,  376,  264],
       device='cuda:0')


# CODING LLM

In [15]:
#19 MILLION PARAMETER
#with 8 batch size it needs 4 gb of gpu memroy
#with 128 batch size it should need 24 gb of gpu memory
#because of the small dataset, resilts will be limited but enough to show fiid improvement during the training and understand the main tech

In [81]:
class GPT(nn.Module):
  def __init__(self):
    super().__init__()
    self.embeddings = nn.Embedding(vocab_size,embed_size)   # 4096 to 384  --> 4096 * 384 -> for each of tokens in total 4096 its gona represent it with the 384 by 1 vector
    self.positions = nn.Embedding(context,embed_size)
    self.blocks = nn.Sequential(*[Block(n_heads) for _ in range(n_layers)])   # this creates 7 blocks and each block has 7 heads
      #  "*" nn.Sequential() expects the modules as separate arguments so * unpacks the 7 blocks seperately , "_" is to say i dont care about the loop variable just repeat it
    self.ln= nn.LayerNorm(embed_size)     # keep numbers on a compfortable range and ti called the normalizatioin
    #each block has attention , computation and normalization and at the end of all the blocks we have conversion of output to actual output form ->
    self.final_linear =  nn.Linear(embed_size,vocab_size,bias=BIAS)  #384*4096

    # the final linear layer produces our logits = our prediction of what are the chances of each one of the vocab
    #tokens to be the next token
    self.apply(self._init_weights)

  def _init_weights(self,module):  #module refers to the linear layer
    if isinstance(module,nn.Linear):  #this asks if the particular layer is linear layer ?
      torch.nn.init.normal_(module.weight,mean=0.0,std=0.02)  #Fill the weight matrix of the current linear layer with random numbers sampled from a normal (Gaussian) distribution.
    #So the weights are randomly initialized around 0 and are generally quite small.#"Whenever you find a Linear layer in my model, initialize all of its weights with small random values drawn from a normal distribution centered at 0."
    #"_" after the zeros means in place
      if module.bias is not None:  #research says when working with LLMs so its better to make the bias terms 0
        torch.nn.init.zeros_(module.bias)

    elif isinstance(module,nn.Embedding): #if embedding layer
      torch.nn.init.normal_(module.weight,mean=0.0,std=0.02)


  def forward(self, input, targets=None):
      # BS = Batch Size / SL = Sequence Length or context length
      # For easier reading, I assume embedding dim of 384 and vocab size of 4096 in comments
      loss= None
      BS,SL = input.shape  # (BS,SL)
      emb = self.embeddings(input)  # (BS,SL,384)
      pos = self.positions(torch.arange(SL, device=device)) # (SL,384)
      x = emb+pos  # combine embedding and positioning stages (BS,SL,384)
      x = self.blocks(x)  #(BS,SL,384)
      x = self.ln(x) # (BS,SL,384)
      logits = self.final_linear(x) # (BS,SL,4096)

        # Calculate Loss if training with targets

        # Cross Entropy Logic
        # (equivalent to negative log likelihood)

        # Information: -log p(x) (inverse of probability)
        # Entropy: avg of information in random variable (prob distribution): - sum_x (x * log(x))
        # CrossEntropy: Compares 2 distr q(true) & p(predicted) in terms of information distance: -sum_x (q(x) * log p(x))
        # LLMs CrossEntropy: true labels are 1 for true, 0 for the rest, so it simplifies to: -sum_x log p(x)

      if targets is not None:
          BS, SL, VS = logits.shape  # (BS,SL,4096)
          logits = logits.view(BS*SL,VS)  # Reshape to prepare for cross_entropy (BS*SL,4096)
          targets = targets.view(BS*SL)   # Reshape as well (BS*SL)
          loss = F.cross_entropy(logits,targets)

            # Optional: Just for fun, manual way to calculate cross_entropy
            # By default, we comment out the manual version to prevent calculating the loss twice (will make things slower)

            # First apply softmax to produce probabilities
            #counts = logits.exp()  # (BS*SL,4096)
            #prob = counts / counts.sum(-1, keepdim=True) # (BS*SL,4096),(BS*SL,1) = (BS*SL,4096)
            #loss2 = -prob[torch.arange(BS*SL),targets].log().mean() # torch.arange(B*T) (BS*SL) | targets (BS*SL)

            # Finally at each of prob's positions, we pick the index specified by the respective target
            # example: targets[3]=329, prob[3][329] = 0.014

            # Most times they will match, sometimes they will not because F.cross_entropy is more precise
            # By uncommenting the following lines, you can see when they don't match
            #if ( not torch.allclose(loss,loss2)):
            #    print(f"[Loss Diff] Pytorch:{loss.item()} Manual:{loss2.item()}")

      return logits,loss



  def generate(self,input, max=500):
    for _ in range(max):
      input=input[:, -context:]  # we are always taking the last 512 tokens of our input   # tensor -> (1,input length untill max of SL)    1 is batch size
      logits, _ = self(input)  #(1,input length,4096)
      logits=logits[:,-1,:]  # we want the model to predict the next toekn afte rthe last token not after every elemwent as last logit is the last position
      probs =F.softmax(logits,dim=-1) #(1,4096)
      #pick one element from the probability distribution
      next=torch.multinomial(probs,num_samples=1)
      input=torch.cat((input,next),dim=1)
    return input




In [47]:

# #optional
# # x,y=get_batch("train")
# # print(x.shape,y.shape)
# # print(x[0][:10])
# # print(y[0][:10])

# model=GPT()
# model=model.to(dtype)
# model=model.to(device)


# @torch.no_grad()
# def generate_sample(input):
#   t1=torch.tensor(encode(input),dtype=torch.long, device=device)#NN works with tensors so tensors:
#   t1=t1.unsqueeze(0) #preserce dimension and add a new at the end so this produce the tensoir : (1,[size of ids])
#   newgen=model.generate(t1,max=64).tolist()
#   results=decode(newgen)
#   print(results)

# generate_sample("once upon a time")

# #logits,loss,loss2=model(x,y)
# # print(loss.item(),loss2.item())  # thiss loss is diff bw the 2 loss distributions .



# CODING THE BLOCKS OF LLM:

In [82]:
class Block(nn.Module):
  def __init__(self,n_heads):
    super().__init__()
    head_size=embed_size // n_heads  #when you enter the transformer block so each token has 384 dim vector embedding so you split them across the heads
    #headsize means whats the dimensionality of wach of the heads in multi head attention mechanism

    self.ma = Multihead(n_heads,head_size)
    self.feed_forward=ForwardLayer(embed_size)# consists of computations
    self.ln1=nn.LayerNorm(embed_size)
    self.ln2=nn.LayerNorm(embed_size)

    # layer norm is going to normailize inputs to that layer across the features for each data point
    # subtract the mean of the points and divide by std dev and then shifting -> more flexible


  def forward(self,x):
    x = x + self.ma(self.ln1(x))    # this stabilizes the computations
    x = x + self.feed_forward(self.ln2(x))
    return x

# Skip connections : they add the input direcctly to the output of certain layers making the training more effective
#they help to prevent  VANISHING GRADIENTS

#NORMALIZATION AS WE WE DONT TWANT THE VALUES NOT TO BE TOO LARGE OR TOO SMALL















In [83]:
class ForwardLayer(nn.Module):
  def __init__(self,embed_size):
    super().__init__()
    self.network=nn.Sequential(
        nn.Linear(embed_size,6*embed_size,bias=BIAS),
        nn.ReLU(),
        nn.Linear(6*embed_size,embed_size,bias=BIAS),
        nn.Dropout(dropout)
    )


  def forward(self,x):
    x = self.network(x)
    return x

# ***TRANSFORMERS AND LLMS ARE the combination of computation and communication . The transforers has computations to detect the patterns in the data and then it has the attention mechanism allow to work with the context -> with relationships bw the differenct parts of the sequence of the data, understand how strong is the relation bw that token and all other tokens***

In [84]:
#multihead attention :


class Multihead(nn.Module):
  def __init__(self,n_heads,head_size):
    super().__init__()
    self.heads=nn.ModuleList([Head(head_size) for _ in range(n_heads)])        # create a list of heads

    #so for Multi head we take the embedding size and split across the all heads
    #so embed=384 and n_heads=7 so head_size = 54 so ---> so 54 dims go to each of the 7 heads

    self.combine =  nn.Linear(head_size*n_heads, embed_size, bias=BIAS)  # 378-> 384

    #so we have 384 dims in the beginning and then they split and after the processign we combine them back to 384 dims

    self.dropout=nn.Dropout(dropout)


  def forward(self,x):
    #run input through all the heads:
    x= torch.cat([head(x) for head in self.heads],dim=-1)  #take input and run all heads and concatenating all results along last dim

    # each head outputs (BS,SL,gead_size)

    x=self.combine(x)  # project to (BS,SL,384)
    x=self.dropout(x)
    return x




In [86]:
class Head(nn.Module):
   def __init__(self,head_size):
    super().__init__()
    self.queries=nn.Linear(embed_size,head_size,bias=BIAS)
    self.keys=nn.Linear(embed_size,head_size,bias=BIAS)
    self.values=nn.Linear(embed_size,head_size,bias=BIAS)

    #in the attention mechanism of llm : query vector represents the word we are focusing on ; key vector represents the all words in the sequence
    #Value vector holds the information of these words. By comparing the query with the keys, model calculates the attention scores hwich is used to
    #weight the values . This process helps the model decide which words to pay more attention to when making the predictions.

    #as the input enters the block so we split each of the tokens of each of the sequences into the different heads


    #so product the 3 projection : 2 of them as queries and keys to find out the compatibility b/w each of the queries with all the keys
    # values projections :->  contain the content that we gonna update with influnece of atention scores that we will have calculated with queries and keys



    self.register_buffer("tril",torch.tril(torch.ones(context,context)))   #register_buffer is a tensor not a model param but still needs to be saved during the model checkpointng. they are used to store fixed statistics
    self.dropout=nn.Dropout(dropout)




   def forward(self,x):

    BS,SL,VS=x.shape
  #q=self.queries(x) --> #so we wanna project the input to from the embedding size of 384 dimensions to vector of 54 vectors for each of tokens

    q=self.queries(x)  # BS,SL,54
    k=self.keys(x)     # BS,SL,54
    v=self.values(x)   # BS,SL,54

  #q IS 512 * 54  AND K ALSO
  #for the dot product the last dim of q and first dim of k should match
    attn_w = q @ k.transpose(-2,-1) * k.shape[-1]**(-0.5)  #BS , SL , SL

  #attention mat be 512 * 512 and 1st rows shows the degree of compatibility of reach token with 1st token and so on .....

  # normalize these matrix multiplication

  #if the result of the dot product is a large number so the vectors are in same direction. IF neg then they are opposite and if 0 so orthogonal
  #vecots are gonna represent the abstract fratires and the meanings of our tokens and words.
  #the QK.T dot product indicates how the each token attends to or is related to all the other tokens

  #but the token should attent to previous tokens before it not after ones so masking is done:

    attn_w=attn_w.masked_fill(self.tril[:SL,:SL]==0, float("-inf"))

    attn_w=F.softmax(attn_w,dim=-1)  #BS,SL,SL

    x= attn_w @ v  # NOW we wanna obtain weighted result where weights are the attention scores bw each token woth 512 other tokens

  #result -> BS, SL ,54

    return x



    attn_w=self.dropout(attn_w)
    out=attn_w @ v
    return out





In [89]:
model=GPT()
model=model.to(dtype)
model=model.to(device)

if compile:
  print("Torch :: Compiling model")
  model=torch.compile(model)

print(sum(p.numel() for p in model.parameters()) / 1e6 , "million params ")

19.837954 million params 


In [90]:
#calculate loss averages

@torch.no_grad()  # Prevent gradient calculation
def calculate_loss():
    out={}
    model.eval()
    for split in ['train','eval']:
        l=torch.zeros(eval_iters)  # Create a tensor of zeros the size of eval_iters
        for i in range(eval_iters):
            x,y=get_batch(split) # Get a new batch of data
            _,loss=model(x,y)  # Calculate the loss
            l[i]=loss  # Store the loss in the next position of tensor
        out[split]=l.mean().item()  # Calculate the mean and extract the final value
    model.train()
    return out

l=calculate_loss()
print(l)

{'train': 8.375, 'eval': 8.375}


# OPTIMIZER AND SCHEDULER :

In [91]:
#################################################################################
# Main Training Process
#################################################################################

# Set Weight Decay differently for different kinds of parameters
# parameter dictionary where keys are parameter names, and values are the parameter themselves
p_dict = {p_name: p for p_name, p in model.named_parameters() if p.requires_grad} # len: 370

# isolate weight matrices as they benefit specially from weight decay
weight_decay_p = [p for n, p in p_dict.items() if p.dim() >= 2]  # len: 171

# isolate other parameters like bias parameters, that don't benefit from weight decay
no_weight_decay_p = [p for n, p in p_dict.items() if p.dim() < 2] # len: 199

# store the parameter types in a list of dictionaries
optimizer_groups = [
    {'params': weight_decay_p, 'weight_decay': weight_decay},
    {'params': no_weight_decay_p, 'weight_decay': 0.0}
]

# Declare optimizer, it helps us compute gradients, update parameters, manage learning rate, apply weight decay
optimizer = torch.optim.AdamW(optimizer_groups, lr=lr, betas=(0.9, 0.99))
# betas: control the exponential moving averages of the gradient and its square,
# which are essential components of the Adam and AdamW optimization algorithms.

# Declare scheduler to change learning rate through the training
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, train_iters, eta_min=lr/10)
# learning rate will descend till a minimum of a tenth of the lr

start_iteration = 0
best_val_loss = float('inf')  # Track best loss value


In [92]:
def load_checkpoint(path):
    print("LLM - Loading model")
    checkpoint = torch.load(path)
    model.load_state_dict(checkpoint['model_state_dict']) # Load parameters
    optimizer.load_state_dict(checkpoint['optimizer_state_dict']) # Load optimizer state
    iteration = checkpoint['iteration'] # In what iteration did we save the model?
    loss = checkpoint['loss'] # What was the last loss value?
    print(f"Loaded iter {iteration} with loss {loss}")
    return iteration, loss

################# OPTIONAL : LOAD A PREVIOUS CHECKPOINT
if os.path.exists(f"{checkpoint_dir}/{checkpoint_load_fn}") and load_pretrained:
    start_iteration, loss = load_checkpoint(checkpoint_dir + checkpoint_load_fn)
    best_val_loss = loss

In [93]:
#inference loop
if inference==True:
    model.eval()
    while True:
         qs = input("Enter text (q to quit) >>> ")
         if qs == "":
             continue
         if qs == 'q':
             break
         generate_sample(qs)

In [ ]:
#training looop
try:
    for i in tqdm(range(start_iteration, train_iters)):
        xb,yb = get_batch("train") # Get a new batch of data
        logits,loss = model(xb,yb) # Run the LLM and get the logits and the loss

        if (i % eval_interval==0 or i == train_iters-1): # Calculate the loss
            l = calculate_loss()
            print(f"\n{i}: train loss: {l['train']} / val loss: {l['eval']}")

            # We do a quick test so that we observe the evolution through the training
            # Remember that we use a very small dataset which doesn't include all topics
            generate_sample("The mountain in my city is") # Generate a sample

            if l['eval'] < best_val_loss: # If we improved the best loss, save a checkpoint
                best_val_loss = l['eval']
                print("[CHECKPOINT]: Saving with loss: ", best_val_loss)
                torch.save({
                    'model_state_dict': model.state_dict(),
                    'optimizer_state_dict': optimizer.state_dict(),
                    'loss': best_val_loss,
                    'iteration': i,
                }, checkpoint_dir + checkpoint_fn)

            if wandb_log:
                wandb.log({
                        "loss/train": l['train'],
                        "loss/val": l['eval'],
                        "lr": scheduler.get_last_lr()[0],
                    },
                    step = i)

        optimizer.zero_grad(set_to_none=True) # Reset gradients
        loss.backward() # Calculate new gradients

        # This line clips the gradients to prevent the exploding gradient problem during training.
        # Exploding gradients can occur when gradients become too large, causing unstable updates to model weights.
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=grad_clip)

        optimizer.step() # Update the model parameters
        scheduler.step() # Update the learning rate value

    if wandb_log:
        wandb.finish()


except KeyboardInterrupt:
    print("Training interrupted. Cleaning up...")

finally:
    # Release GPU memory
    torch.cuda.empty_cache()
    print("GPU memory released.")

if wandb_log:
    wandb.finish()
torch.cuda.empty_cache()







  0%|          | 0/100000 [00:00<?, ?it/s]


0: train loss: 8.375 / val loss: 8.375
['The mountain in my city is Bang Mos balllamnown officeends Jeanant with Angelificationfeck each Mil architectbedyacedaugh project novelistr will var noturies fourth word�aps thought arg Elect outside usuallyights Gram Chiefba� geetworkball throughrew studied dev Notudi university defe sim crit Sirhes stand womenerv� Ver term him']
[CHECKPOINT]: Saving with loss:  8.375


  0%|          | 50/100000 [00:19<8:55:07,  3.11it/s]


50: train loss: 6.087500095367432 / val loss: 6.131249904632568
['The mountain in my city is college worked sy Club each areferalledistsate Os Olympics on he cell from 3 thatomenavi, inf be Valley was coloninmxt.\n\n\n\n Cupot said De Footballne th Award in R Sch7ia known dis Modations Schoolastern tks Cro. Rs from born is']
[CHECKPOINT]: Saving with loss:  6.131249904632568


  0%|          | 100/100000 [00:38<8:28:44,  3.27it/s]

...

  3%|▎         | 3200/100000 [19:47<8:09:07,  3.30it/s]


3200: train loss: 3.7265625 / val loss: 3.871875047683716


  3%|▎         | 3201/100000 [19:50<31:54:28,  1.19s/it]

['The mountain in my city is used in a population of 338 people in the city.\n\n\n\nToilia Province\n\nTakiao (12 October 1953 – 4 July 201) was a Valvadian professional wrestlerildargements between the']


  3%|▎         | 3228/100000 [19:58<8:09:30,  3.29it/s]